# ResRank stage 2 (English E2Rank), starting from the stage-1 snapshot

Works on **one T4** or **T4 x2** (detected in cell 1). With two GPUs, stage 2 trains on GPU 0 and snapshots are
evaluated on GPU 1 while it runs. With one GPU, evaluation waits until training is stopped or finished (training
peaks at about 15 GB of the T4's 15.6 GB, so nothing else fits next to it).

Order: stop stage 1 and keep its newest snapshot as `stage1_final` -> free disk -> measure stage 1 on SciFact and
NFCorpus -> upload `stage1_final` once -> pick the English E2Rank sources -> write the config -> short timing run on
stage-2 data -> plan the sample count -> build the file -> train in the background -> evaluate.

**New session?** Run cell 0 first. Then run everything from cell 1 in order: every cell skips work that is already
done, and the plan saved with the checkpoints makes a resumed session rebuild exactly the same training file.

**Set `HOURS_LEFT` in cell 1 before anything else.** Requires the newest code zip as the code dataset (cell 0 checks).

In [ ]:
%%bash
# 0. ONLY IN A NEW SESSION: copy the code, install, restore stage1_final and any stage-2 progress
rm -rf /kaggle/working/resrank
cp -r /kaggle/input/datasets/tanmaysarda1/it550-final-project/resrank /kaggle/working/resrank
cd /kaggle/working/resrank
grep -q publish_init src/resrank/train/run_training.py && grep -q suggest_english_sources src/resrank/data/e2rank.py \
  || { echo "OLD CODE in the code dataset: upload the newest zip as a new version of it, then rerun this cell"; exit 1; }
pip install -q -e .
pip install -q transformers huggingface_hub pyyaml pytest bitsandbytes kagglehub
python -m pytest -q -x 2>&1 | tail -1
mkdir -p /kaggle/working/checkpoints
S1=/kaggle/input/datasets/tanmaysarda1/resrank-stage1-final
CK=/kaggle/input/datasets/tanmaysarda1/resrank-checkpoints
[ -d $S1/stage1_final ] && [ ! -d /kaggle/working/checkpoints/stage1_final ] && cp -r $S1/stage1_final /kaggle/working/checkpoints/ && echo "restored stage1_final"
[ -d $CK/stage2_e2rank ] && [ ! -d /kaggle/working/checkpoints/stage2_e2rank ] && cp -r $CK/stage2_e2rank /kaggle/working/checkpoints/ && echo "restored stage2_e2rank (resume point and plan)"
ls /kaggle/working/checkpoints

In [ ]:
# 1. Token, environment, budget, GPUs
import os, sys, json, pathlib, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
sys.path.insert(0, "/kaggle/working/resrank/src")

HOURS_LEFT = 10.0                         # <-- SET THIS: GPU hours you have left this week
EVAL_DATASETS = ["scifact", "nfcorpus"]   # small BEIR sets: evaluated, and their queries are removed from training

N_GPU = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
os.environ["N_GPU"] = str(N_GPU)
os.environ["EVAL_GPU"] = "1" if N_GPU >= 2 else "0"
os.environ["EVAL_DATASETS"] = " ".join(EVAL_DATASETS)
WORK, CKD = pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/working/checkpoints")
print("token loaded:", bool(os.environ.get("KAGGLE_API_TOKEN")), "| HOURS_LEFT =", HOURS_LEFT, "| GPUs:", N_GPU)
if N_GPU < 2:
    print("one GPU: snapshots cannot be evaluated while stage 2 trains (cell 12 waits for training to stop)")

In [ ]:
%%bash
# 2. Stop stage 1 (only stage 1: a stage-2 run is left alone) and keep its newest complete snapshot as stage1_final
pkill -f "[s]tage1_small.yaml" && sleep 5; pgrep -af "[s]tage1_small.yaml" || echo "no stage-1 process running"
cd /kaggle/working/checkpoints
if [ -d stage1_final ]; then echo "stage1_final already exists, keeping it"; else
  [ -f stage1_full/latest/train_state.json ] || { echo "NO COMPLETE SNAPSHOT in stage1_full/latest; stop here and check 'ls stage1_full'"; exit 1; }
  mv stage1_full/latest stage1_final
fi
echo "stage1_final comes from:"; cat stage1_final/train_state.json; echo
ls stage1_final

In [ ]:
%%bash
# 3. Free disk: stage 1 data and leftovers are no longer needed (the working folder is limited to about 19.5 GB;
#    stage1_final + stage-2 snapshot + stage-2 final are 3 x 4.8 GB)
rm -rf /kaggle/working/data/01_raw/pe_rank /kaggle/working/data/02_train/stage1.jsonl /kaggle/working/data/02_train/timing.jsonl
rm -rf /kaggle/working/checkpoints/timing_run /kaggle/working/checkpoints/stage1 /kaggle/working/checkpoints/snap_step* \
       /kaggle/working/checkpoints/s2_step* /kaggle/working/_upload_stage
df -h /kaggle/working | tail -1
du -sh /kaggle/working/data/* /kaggle/working/checkpoints/* 2>/dev/null

In [ ]:
%%bash
# 4. Evaluation helpers (written to a file so cells 5 and 12 share them), then measure stage 1 on every EVAL_DATASET.
#    Runs on GPU 0 before stage 2 starts. About 10 minutes for SciFact + NFCorpus. Existing runs are reused.
cat > /kaggle/working/eval_lib.sh <<'EOF'
R=/kaggle/working/data; CKD=/kaggle/working/checkpoints
prepare () {   # $1 = dataset: download it and build BM25 candidates for its test queries (once)
  local D=$R/01_raw/beir/$1
  [ -f $R/03_candidates/$1.bm25.run ] && return 0
  python -m resrank.data.s01_download_beir --name $1 > /dev/null
  python -m resrank.data.s03_bm25 --dataset $1 --corpus $D/corpus.jsonl --queries $D/queries.jsonl --qrels $D/qrels/test.tsv
}
run_one () {   # $1 = dataset, $2 = checkpoint name, $3 = GPU, $4 = orders (default: original)
  local D=$R/01_raw/beir/$1 ORDERS=${4:-original}
  [ -f $R/05_runs/$1.$2.${ORDERS%% *}.run ] && return 0
  CUDA_VISIBLE_DEVICES=$3 python -m resrank.eval.s06_encode_corpus --dataset $1 --corpus $D/corpus.jsonl --ckpt $2 --only-candidates $R/03_candidates/$1.bm25.run
  CUDA_VISIBLE_DEVICES=$3 python -m resrank.eval.s07_rerank --dataset $1 --queries $D/queries.jsonl --ckpt $2 --orders $ORDERS --batch-size 4
}
score () {     # $1 = dataset: nDCG@10 of BM25 and of every run so far
  local ARGS="--run bm25=$R/03_candidates/$1.bm25.run" f L
  for f in $R/05_runs/$1.*.run; do L=$(basename $f .run); L=${L#$1.}; ARGS="$ARGS --run $L=$f"; done
  python -m resrank.eval.s08_evaluate --dataset $1 --qrels $R/01_raw/beir/$1/qrels/test.tsv --name ndcg_$1 $ARGS | grep "{'dataset'"
}
EOF
if [ -f /kaggle/working/checkpoints/stage2_e2rank/plan.json ]; then
  echo "stage 1 was measured in an earlier session; cell 13 measures it again next to stage 2"; exit 0; fi
cd /kaggle/working/resrank && source /kaggle/working/eval_lib.sh
for DS in $EVAL_DATASETS; do prepare $DS; run_one $DS stage1_final 0; score $DS; done

In [ ]:
%%bash
# 5. Upload stage1_final to its OWN dataset, in the background, ONCE (each upload is a new 4.8 GB version that counts
#    against the 200 GB quota). Skipped if the dataset already holds it (it was attached in cell 0).
if [ -d /kaggle/input/datasets/tanmaysarda1/resrank-stage1-final/stage1_final ]; then
  echo "stage1_final is already in tanmaysarda1/resrank-stage1-final: not uploading again"; exit 0; fi
if pgrep -f "[r]esrank-stage1-final" > /dev/null; then echo "upload already running"; exit 0; fi
cd /kaggle/working/resrank
setsid nohup python -c "
from resrank.train.publish import upload_checkpoints
n = upload_checkpoints('/kaggle/working/checkpoints', 'tanmaysarda1/resrank-stage1-final', ['stage1_final'],
                       version_notes='stage-1 snapshot', stage_dir='/kaggle/working/_upload_stage_s1')
print('uploaded', n, 'files')
" > /kaggle/working/upload_stage1.log 2>&1 < /dev/null &
sleep 5
echo "upload started in the background; cell 11 shows its status"

In [ ]:
# 6. Download E2Rank and pick the English sources (CPU only, about 3-5 minutes; skipped when the plan already exists).
#    The script filter rejects CJK, Cyrillic, Arabic, Thai, Indic... but Finnish, Indonesian, Swahili, Spanish, French
#    (Mr.TyDi / MIRACL) are Latin script and pass it. The 'english_words' column catches those.
from resrank.data.e2rank import e2rank_stats, source_report, suggest_english_sources

PLAN = CKD / "stage2_e2rank" / "plan.json"
E2 = WORK / "data/01_raw/e2rank/train.jsonl"
if PLAN.exists():
    plan = json.loads(PLAN.read_text())
    ALLOW_SOURCES = plan["allow_sources"]
    print("saved plan found: using its sources:", ALLOW_SOURCES)
else:
    if not E2.exists():
        subprocess.run(["python", "-m", "resrank.data.s01_download", "e2rank"], cwd="/kaggle/working/resrank", check=True)
    print(json.dumps(e2rank_stats(E2, limit=20000), indent=1))
    report = source_report(E2)                       # all rows counted; text checks on 1,000 rows per source
    print(f"\n{'source':32s} {'rows':>7s} {'pass_script':>11s} {'english_words':>13s}")
    for s, r in report.items():
        print(f"{s:32s} {r['count']:7d} {r['frac_pass_script_filter']:11.3f} {r['english_word_rate']:13.3f}")
    ALLOW_SOURCES = suggest_english_sources(report)
    ALLOW_SOURCES_OVERRIDE = None   # e.g. ["msmarco", "nq", ...] if you disagree with the suggestion after reading the table
    if ALLOW_SOURCES_OVERRIDE:
        ALLOW_SOURCES = sorted(ALLOW_SOURCES_OVERRIDE)
    AVAILABLE = int(sum(report[s]["count"] * report[s]["frac_pass_script_filter"] for s in ALLOW_SOURCES))
    print("\nENGLISH SOURCES:", ALLOW_SOURCES, f"\nabout {AVAILABLE} usable samples")
os.environ["ALLOW_SOURCES"] = " ".join(ALLOW_SOURCES)

In [ ]:
# 7. Write the stage-2 config. publish_init: false keeps stage1_final out of the checkpoints dataset (it has its own).
import yaml
cfg = {"train_file": "stage2_e2rank_en", "out_name": "stage2_e2rank", "init_from": "stage1_final", "epochs": 1,
       "micro_batch_size": 1, "grad_accum": 16, "lr": 6.0e-6, "warmup_ratio": 0.03, "weight_decay": 0.0,
       "max_grad_norm": 1.0, "precision": "fp16", "init_loss_scale": 1024, "optimizer": "adamw8bit", "log_every": 1,
       "save_every": 0, "save_every_minutes": 15, "max_steps": None,
       "publish_dataset": "tanmaysarda1/resrank-checkpoints", "publish_every_saves": 4,
       "publish_include": [], "publish_init": False}
pathlib.Path("/kaggle/working/resrank/configs/stage2_run.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(open("/kaggle/working/resrank/configs/stage2_run.yaml").read())

In [ ]:
%%bash
# 8. Timing run on STAGE-2 data (about 6 minutes on GPU 0): E2Rank passages differ in length from PE-Rank, so the
#    stage-1 speed is only a fallback. Skipped when the plan already exists. Nothing is saved or uploaded.
[ -f /kaggle/working/checkpoints/stage2_e2rank/plan.json ] && { echo "saved plan found: skipping the timing run"; exit 0; }
cd /kaggle/working/resrank
python -m resrank.data.s02_build_train stage2 --input /kaggle/working/data/01_raw/e2rank/train.jsonl \
  --name stage2_timing --n 192 --n-candidates 16 --seed 7 --allow-sources $ALLOW_SOURCES > /dev/null
CUDA_VISIBLE_DEVICES=0 python -u -m resrank.train.run_training --train-config configs/stage2_run.yaml \
  --train-file stage2_timing --out-name timing_s2 --no-publish --no-save 2>&1 | grep -vE "^\s*$" | tail -3
python -m resrank.train.log_summary /kaggle/working/checkpoints/timing_s2

In [ ]:
# 9. Plan the sample count from the measured stage-2 speed and HOURS_LEFT. Saved next to the checkpoints, so a resumed
#    session rebuilds exactly the same file (the build is deterministic given these settings).
from resrank.train.log_summary import summarize
from resrank.train.plan import plan_samples

N_OVERRIDE = None       # e.g. 12000 to choose your own; ignored when a saved plan exists
STAGE1_SEC_PER_SAMPLE = 1.52   # median of the stage-1 log (24.3 s per 16-sample step); fallback only
if PLAN.exists():
    plan = json.loads(PLAN.read_text())
    print("reusing the saved plan (it must not change, or --resume will refuse to continue):")
else:
    timing = CKD / "timing_s2"
    if (timing / "train_log.jsonl").exists():
        s = summarize(timing)
        sps, source = s["sec_per_sample_incl_warmup"], "stage-2 timing run"
        if s["oom_skipped_steps"]:
            print("WARNING: the timing run hit out-of-memory; lower model.max_passage_len before training")
    else:
        sps, source = STAGE1_SEC_PER_SAMPLE, "stage-1 log (fallback)"
    # reserve: evaluation 1.0 h (1.5 h with one GPU, where it cannot overlap training), setup 0.5 h, restarts 1.5 h
    reserve = 3.0 if N_GPU >= 2 else 3.5
    plan = plan_samples(sps, HOURS_LEFT, reserve_hours=reserve, max_samples=max(16, int(0.95 * AVAILABLE) // 1000 * 1000))  # 5% margin: the pass rate is estimated
    if N_OVERRIDE:
        plan.update(n_samples=N_OVERRIDE, optimizer_steps=N_OVERRIDE // 16, note="manual override")
    plan.update(sec_per_sample_source=source, allow_sources=ALLOW_SOURCES, exclude_datasets=EVAL_DATASETS, seed=42)
    PLAN.parent.mkdir(parents=True, exist_ok=True)
    PLAN.write_text(json.dumps(plan))
    subprocess.run(["rm", "-rf", str(timing), str(WORK / "data/02_train/stage2_timing.jsonl")])
print(json.dumps(plan, indent=2))
os.environ["N_SAMPLES"] = str(plan["n_samples"])
os.environ["EXCLUDE_DATASETS"] = " ".join(plan["exclude_datasets"])

In [ ]:
%%bash
# 10. Build the stage-2 training file from the plan: English sources only, the evaluation queries removed.
#     Then delete the raw E2Rank file (1.8 GB of the tight disk); a new session downloads it again in cell 6.
cd /kaggle/working/resrank
OUT=/kaggle/working/data/02_train/stage2_e2rank_en.jsonl
if [ -f $OUT ] && [ "$(wc -l < $OUT)" = "$N_SAMPLES" ]; then echo "training file already built ($N_SAMPLES samples)"; exit 0; fi
EXCL=""; for DS in $EXCLUDE_DATASETS; do
  [ -f /kaggle/working/data/01_raw/beir/$DS/queries.jsonl ] || python -m resrank.data.s01_download_beir --name $DS > /dev/null
  EXCL="$EXCL /kaggle/working/data/01_raw/beir/$DS/queries.jsonl"; done
[ -f /kaggle/working/data/01_raw/e2rank/train.jsonl ] || python -m resrank.data.s01_download e2rank
python -m resrank.data.s02_build_train stage2 --input /kaggle/working/data/01_raw/e2rank/train.jsonl \
  --name stage2_e2rank_en --n $N_SAMPLES --n-candidates 16 --allow-sources $ALLOW_SOURCES --exclude-queries $EXCL > /dev/null || exit 1
python -c "
import json; s = json.load(open('/kaggle/working/data/02_train/stage2_e2rank_en.stats.json'))
print(json.dumps({k: v for k, v in s.items()}, indent=1))
assert s['written'] == $N_SAMPLES, 'fewer usable samples than planned: lower N_OVERRIDE in cell 9 (delete plan.json first)'
" || exit 1
rm -f /kaggle/working/data/01_raw/e2rank/train.jsonl
df -h /kaggle/working | tail -1

In [ ]:
%%bash
# 11. Start stage 2 in the background on GPU 0 (fresh, resumes from a snapshot, or does nothing if finished)
cd /kaggle/working/resrank
if pgrep -f "[s]tage2_run.yaml" > /dev/null; then echo "stage 2 is already running"; exit 0; fi
CUDA_VISIBLE_DEVICES=0 setsid nohup python -u -m resrank.train.run_training --train-config configs/stage2_run.yaml --auto-resume >> /kaggle/working/train2.log 2>&1 < /dev/null &
echo "launched; waiting 90 s to catch start-up errors (loading both models takes about a minute)..."
sleep 90
echo "--- process (empty means it already stopped) ---"; pgrep -af "[s]tage2_run.yaml" | cut -c1-90
echo "--- end of /kaggle/working/train2.log ---"
tail -c 1500 /kaggle/working/train2.log | tr '\r' '\n' | grep -v "^$" | tail -6

In [ ]:
# 12. Monitor: progress, recent losses (100-step means), GPUs, newest snapshot, the stage-1 upload, disk
import statistics as st
print(subprocess.run("tail -c 600 /kaggle/working/train2.log | tr '\\r' '\\n' | grep -E 'training:|WARNING|publish|finished|Error|Traceback|already' | tail -3",
                     shell=True, capture_output=True, text=True).stdout)
log_path = CKD / "stage2_e2rank/train_log.jsonl"
if log_path.exists():
    recs = [r for r in map(json.loads, log_path.read_text().splitlines()) if not r.get("oom_skipped")]
    for i in range(0, len(recs), 100):
        w = recs[i:i + 100]
        print(f"steps {w[0]['step']:5d}-{w[-1]['step']:5d}  ranknet {st.mean(r['ranknet'] for r in w):.4f}  "
              f"infonce {st.mean(r['infonce'] for r in w):.4f}  lr {w[-1]['lr']:.1e}")
print(subprocess.run("pgrep -af '[s]tage2_run.yaml' | cut -c1-80; nvidia-smi --query-gpu=index,memory.used,memory.total,utilization.gpu --format=csv,noheader; "
                     "cat /kaggle/working/checkpoints/stage2_e2rank/latest/train_state.json 2>/dev/null; echo; "
                     "tail -c 200 /kaggle/working/upload_stage1.log 2>/dev/null; df -h /kaggle/working | tail -1",
                     shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%bash
# 13. Evaluate the newest stage-2 weights on every EVAL_DATASET, next to BM25 and stage1_final.
#     Two GPUs: runs on GPU 1 while stage 2 trains. One GPU: only when training is not running.
#     Once stage 2 has finished, this evaluates the final weights, also with reversed and random input order (Table VI).
cd /kaggle/working/resrank && source /kaggle/working/eval_lib.sh
if [ "$N_GPU" -lt 2 ] && pgrep -f "[s]tage2_run.yaml" > /dev/null; then
  echo "one GPU and stage 2 is training: stop it first (cell 15) or wait until it finishes"; exit 0; fi
if [ -d $CKD/stage2_e2rank/encoder ]; then SRC=stage2_e2rank; ORDERS="original reversed random"
else SRC=stage2_e2rank/latest; ORDERS=original; fi
[ -f $CKD/$SRC/train_state.json ] || { echo "no stage-2 weights yet"; exit 0; }
STEP=$(python -c "import json; print(json.load(open('$CKD/$SRC/train_state.json'))['step'])")
SNAP=s2_step$STEP
rm -rf $CKD/$SNAP; cp -al $CKD/$SRC $CKD/$SNAP   # hard links: no extra disk, stable even if training replaces 'latest'
rm -rf $CKD/$SNAP/latest                        # the final folder also holds latest/; the copy does not need it
echo "evaluating stage-2 weights from step $STEP ($SRC) on GPU $EVAL_GPU"
for DS in $EVAL_DATASETS; do prepare $DS; run_one $DS stage1_final $EVAL_GPU; run_one $DS $SNAP $EVAL_GPU "$ORDERS"; score $DS; done
rm -rf $CKD/$SNAP

In [ ]:
# 14. Is stage 2 really better than stage 1? Paired bootstrap over each dataset's test queries (95% interval).
import glob
from resrank.io_utils import read_trec_run, read_qrels
from resrank.eval.metrics import evaluate_run, paired_bootstrap
R = "/kaggle/working/data"
for DS in EVAL_DATASETS:
    runs = sorted(glob.glob(f"{R}/05_runs/{DS}.s2_step*.original.run"), key=lambda p: int(p.split("s2_step")[1].split(".")[0]))
    if not runs:
        print(DS, ": no stage-2 run yet (cell 13)"); continue
    qrels = read_qrels(f"{R}/01_raw/beir/{DS}/qrels/test.tsv")
    A, B = runs[-1], f"{R}/05_runs/{DS}.stage1_final.original.run"
    per_query = lambda path: evaluate_run(read_trec_run(path), qrels, "ndcg", 10)["per_query"]
    a, b = per_query(A), per_query(B)
    qs = sorted(qrels)
    res = paired_bootstrap([a[q] for q in qs], [b[q] for q in qs], n_boot=10000, seed=0)
    verdict = ("stage 2 better" if res["ci_low"] > 0 else "stage 1 better" if res["ci_high"] < 0
               else "no clear difference (95% interval includes 0)")
    print(f"{DS}: {os.path.basename(A)} vs stage1_final  {({k: round(v, 4) for k, v in res.items()})}  -> {verdict}")

In [ ]:
%%bash
# 15. Stop stage 2 (the newest snapshot stays; rerun cell 11 to resume from it)
pkill -f "[s]tage2_run.yaml"; sleep 5; pgrep -af "[s]tage2_run.yaml" || echo "stopped"